# Tests physique front-tracking

## Introduction
 
 Validation made by: B.M. & G.B. 
 
 Report generated 03/03/2025

### Description
 
Verification of the implementation of the gravity source term. Without viscosity, the drop should fall with a constant acceleration and the trajectory is $y=0.06-0.5 \cdot 10 x^2$ 

### TRUST parameters
 
* Version TRUST :  1.9.6


In [ ]:
from trustutils import run

run.reset() 
dts = [0.001, 0.01, 0.03, 0.1]
for dt in dts:
   run.addCaseFromTemplate("template.data",f"DT{dt}",{"dtmax":dt }, nbProcs=1, targetData="prepare.data")

#run.executeCommand("tar xzf sauv_lata.tgz",verbose=1)
#run.executeCommand("sed -i \"/rank_reprise_temperature/d\" temperature_channel_monop.data")
run.printCases()
run.runCases() # Mettre preventConcurrent si besoin


In [ ]:
%%bash
cd build
for d in $(ls -d DT*)
do
    cd $d
    awk '$1=="Centre_gravite_phases"{print $4, $6,$7,$8}' prepare.err >position_goutte.txt
    cd ..
done


## Vitesse de chute libre sous gravite, VEF
 
Chute libre d'une goutte (on doit observer une acceleration de $-10m/s^2$)

3 plots are shown: 
 - Sonde de vitesse dans la goutte en fonction du temps
 - Coordonnee Z du centre de gravite de la phase 0 (un ecart important est normal, du a integration par Euler explicite)
 - Definition: Erreur = (vitesse calculee 	 - vitesse analytique) / vitesse\_analytique
 
As the timestep is refined, the approximation of the analytical solution is improved. 
As we use an Euler Explicit scheme, the error ov a timestep converges as $\mathcal{O}(\Delta t)$, so that at a final time $t_f=n \Delta t$, the error accumulates. As the error decrease with the timestep, but the number of timestep to reach $t_f$ increases as $t_f/\Delta t$; it is then expected to reach the same relative error at the end of the simulation for $dt= 10$ms and 1ms. 
\begin{equation}
e_f(\Delta t) = \sum_n e(\Delta t) = n e(\Delta t) = \frac{t_f}{\Delta t}\mathcal{O}(\Delta t) = \mathcal{O}(1)
\end{equation}

To get a time convergence of the velocity, a higher order-scheme as RK3_FT is required

In [ ]:
import pandas as pd
import numpy as np
import os
from trustutils import plot 

tmax = 0.07
import numpy as np 
time = np.linspace(0.0,tmax,1000) 
y_ana = 0.06 - 0.5*10*time*time
vy_ana = -10*time
fig_velo = plot.Graph(r"Drop velocity") 
fig_velo.add(time,vy_ana,label=r"analytique",marker='-')
fig_velo.label(r"s",r"m/s")

fig = plot.Graph(r"Position du centre de la goutte") 
fig.add(time,y_ana,label=r"theorique",marker='-') 
fig.label(r"s",r"m")

fig3 = plot.Graph(r"Relative error on drop velocity") 
# plot theoretical error (=0 obviously) first to get the same colors on each figure
fig3.add(time,0*time,label=r"theorique",marker='-') 
fig3.label(r"s",r"m/s")

positions = {}
for dt in dts:
    fic = os.path.join(run.BUILD_DIRECTORY, f"DT{dt}","position_goutte.txt")
    pos = plot.loadText(fic)
    son = os.path.join(run.BUILD_DIRECTORY, f"DT{dt}","prepare_V_GOUTTE.son")
    data = plot.loadText(son)
    if os.path.isfile(fic):
        mat =  np.loadtxt(fic).T
        t,x,y,z = mat
        print(f"dt {dt} ok")
        positions[dt] = mat
        pass
    data = plot.loadText(son)
    lab=r"$"+f"dt={dt}$s"
    fig_velo.add(data[0],data[3],label=lab,marker='-x')
    fig.add(pos[0],pos[3],label=lab,marker='-x')
    fig3.add(data[0],((data[3]+10*data[0])/(10*data[0]+1e-9)),label=lab,marker='-x')
    

#print(positions[0.01][0,:])
#for dirpath, dirnames, filenames in os.walk("."):
#   for filename in filenames:
#      if filename.endswith(".txt"):
#         print(dirpath, filename)

# pd.DataFrame(data=None, index=None, columns=None, dtype=None, copy=None)[source]


### indicatrice

In [ ]:
from trustutils import visit
 
visu = visit.Show("./DT0.01/lata/prepa.lata","Mesh","INTERFACES")
visu.addField("./DT0.01/lata/prepa.lata","Pseudocolor","INDICATRICE_INTERF_ELEM_DOM",mesh="DOM")
visu.slice(origin=[0.,0.,0.],normal=[0.,-1.,0],type_op="slice")
visu.normal3D([-0.5,0.7,0.4])
visu.up3D([0.2,-0.3,0.9])
visu.plot()

### vitesse
 
 

In [ ]:
from trustutils import visit
 
visu = visit.Show("./DT0.01/lata/prepa.lata","Pseudocolor","VITESSE_Z_FACES_DOM_dual",mesh="DOM_dual")
visu.slice(origin=[0.,-0.000001,0.],normal=[0.,-1.,0],type_op="slice")
visu.addField("./DT0.01/lata/prepa.lata","Vector","VITESSE_FACES_DOM_dual",mesh="DOM_dual")
visu.slice(origin=[0.,0.,0.],normal=[0.,-1.,0],type_op="slice")
visu.normal3D([0.,-1.,0.])
visu.up3D([0.,0.,1.])
visu.plot()